In [8]:
import boto3
import sagemaker
from sagemaker.sklearn.estimator import SKLearn


# Create a boto3 session in us-east-2
boto_session = boto3.Session(region_name="ca-central-1")
sagemaker_session = sagemaker.Session(boto_session=boto_session)

role = "arn:aws:iam::222634404112:role/SageMakerExecutionRole-ca"
job_name = "occupancy-model-t1"


[04/09/25 22:49:22] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=425121;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=313143;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1352\1352]8;;\

In [9]:
sklearn_estimator = SKLearn(
    entry_point="train.py",       # Use your training script
    source_dir=".",               # Package the current directory (which includes train.py, inference.py, requirements.txt, etc.)
    dependencies=["requirements.txt"],
    role=role,
    instance_type="ml.m5.xlarge",
    framework_version="0.23-1",
    output_path="s3://dana-minicapstone-ca/model-artifacts/",
    sagemaker_session=sagemaker_session,
    job_name=job_name
)

sklearn_estimator.fit()


                    INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=41833;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=194389;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[04/09/25 22:49:29] INFO     Creating training-job with name:                                       ]8;id=1034;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=900689;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py#1042\1042]8;;\
                             sagemaker-scikit-learn-2025-04-10-05-49-22-842                                        

2025-04-10 05:49:31 Starting - Starting the training job...
2025-04-10 05:50:02 Downloading - Downloading input data...
2025-04-10 05:50:17 Downloading - Downloading the training image...
2025-04-10 05:50:53 Training - Training image download completed. Training in progress..2025-04-10 05:51:00,720 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2025-04-10 05:51:00,723 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-04-10 05:51:00,762 sagemaker_sklearn_container.training INFO     Invoking user training script.
2025-04-10 05:51:01,005 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt:
/miniconda3/bin/python -m pip install -r requirements.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 84.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.9/255.9 MB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 27.5 MB/s eta 0:00:0

In [10]:
# from sagemaker.sklearn.model import SKLearnModel

# # Use the model_data from your training job
# model = SKLearnModel(
#     model_data=sklearn_estimator.model_data,
#     role=role,
#     entry_point="inference.py",  # Reference your inference script
#     framework_version="0.23-1",
#     py_version="py3",
#     source_dir="."  # Include any additional files if needed
# )

# predictor = model.deploy(initial_instance_count=1, instance_type="ml.m5.large")


In [11]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serializers import CSVSerializer

model = SKLearnModel(
    model_data=sklearn_estimator.model_data,
    role=role,
    entry_point="inference.py",  # Reference your inference script
    framework_version="0.23-1",
    py_version="py3",
    source_dir="."
)

predictor = model.deploy(initial_instance_count=1, instance_type="ml.m5.large")
# Set the serializer to CSV so that payload is sent as text/csv
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()


# Retrieve test CSV from S3 and then call the endpoint
s3 = boto3.client('s3', region_name='ca-central-1')
bucket_name = "dana-minicapstone-ca"
test_key = "data/hvac_test.csv"
response = s3.get_object(Bucket=bucket_name, Key=test_key)
test_csv = response['Body'].read().decode('utf-8')

# Call your endpoint with the CSV string
result = predictor.predict(test_csv)
print("Predictions from endpoint:\n", result)


[04/09/25 22:52:01] INFO     Creating model with name:                                              ]8;id=272887;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=991433;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py#4094\4094]8;;\
                             sagemaker-scikit-learn-2025-04-10-05-52-01-763                                        

[04/09/25 22:52:02] INFO     Creating endpoint-config with name                                     ]8;id=233039;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=84308;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py#6019\6019]8;;\
                             sagemaker-scikit-learn-2025-04-10-05-52-02-733                                        

[04/09/25 22:52:03] INFO     Creating endpoint with name                                            ]8;id=43502;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=66980;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py#4841\4841]8;;\
                             sagemaker-scikit-learn-2025-04-10-05-52-02-733                                        

------!Predictions from endpoint:
 [['Predicted_HVAC_kWh'], ['2.3969830568836987'], ['2.3970483834481318'], ['2.3971137100125643'], ['2.3971790365769974'], ['2.3972443631414304'], ['0.06740179557134152'], ['0.06746712213577455'], ['0.06753244870020758'], ['0.06759777526464061'], ['0.0676631018290732'], ['0.06772842839350623'], ['0.06779375495793927'], ['0.06785908152237186'], ['0.06792440808680489'], ['0.4206308403030934'], ['0.30037465464523594'], ['0.26343336461726174'], ['1.141708043674981'], ['0.07904091212478948'], ['0.07902708372096079'], ['6.007891324241083'], ['5.147318509173434'], ['4.35818019193904'], ['3.9875536230185484'], ['2.995998170545506'], ['3.5240212941604145'], ['6.073884632504633'], ['5.288195278435521'], ['5.532615568302566'], ['5.464555170074519'], ['5.444094087489781'], ['5.456673290015518'], ['6.083860780352534'], ['6.767309094893041'], ['7.680001641611282'], ['8.255294228765315'], ['8.548657322968907'], ['8.742005730588028'], ['8.55467500765295'], ['8.55474033